In [17]:
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf
import matplotlib.pyplot as plt

from keras.preprocessing.image import ImageDataGenerator
from keras.applications import MobileNetV2
from keras.layers import Dense, GlobalAveragePooling2D
from keras.models import Model

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.15.0


In [18]:
IMG_SIZE = 224

BATCH_SIZE = 32

DATASET_PATH = "../dataset"

train_datagen = ImageDataGenerator(

    rescale=1./255,

    validation_split=0.2,

    rotation_range=20,

    zoom_range=0.2,

    horizontal_flip=True

)

In [19]:
train_data = train_datagen.flow_from_directory(

    DATASET_PATH,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    subset='training'

)

val_data = train_datagen.flow_from_directory(

    DATASET_PATH,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    subset='validation'

)

Found 16516 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.


In [20]:
base_model = MobileNetV2(

    weights='imagenet',

    include_top=False,

    input_shape=(224,224,3)

)

base_model.trainable = False

In [21]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

predictions = Dense(

    train_data.num_classes,

    activation='softmax'

)(x)

model = Model(

    inputs=base_model.input,

    outputs=predictions

)

In [22]:
model.compile(

    optimizer='adam',

    loss='categorical_crossentropy',

    metrics=['accuracy']

)

In [23]:
history = model.fit(

    train_data,

    validation_data=val_data,

    epochs=5

)

Epoch 1/5


517/517 [==============================] - 721s 1s/step - loss: 0.6338 - accuracy: 0.7990 - val_loss: 0.3970 - val_accuracy: 0.8702
Epoch 2/5
517/517 [==============================] - 731s 1s/step - loss: 0.3548 - accuracy: 0.8818 - val_loss: 0.3352 - val_accuracy: 0.8823
Epoch 3/5
517/517 [==============================] - 717s 1s/step - loss: 0.2976 - accuracy: 0.8986 - val_loss: 0.2982 - val_accuracy: 0.8976
Epoch 4/5
517/517 [==============================] - 692s 1s/step - loss: 0.2576 - accuracy: 0.9119 - val_loss: 0.3061 - val_accuracy: 0.8901
Epoch 5/5
517/517 [==============================] - 738s 1s/step - loss: 0.2456 - accuracy: 0.9169 - val_loss: 0.2537 - val_accuracy: 0.9134


In [24]:
model.save("../models/crop_disease_model.keras")